In [1]:
import mlflow
import numpy as np
import pandas as pd

import dataprocessing as dc


In [2]:
#Uncomment to create the csv

# df = pd.read_table('data/SMSSpamCollection', header=None, names=['Category', 'Message'])

# # Save to CSV format
# df.to_csv('data/SMSSpamCollection.csv', index=False, encoding='utf-8')

In [3]:
datafile = "SMSSpamCollection.csv"

df = pd.read_csv(f"data/{datafile}")
newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = np.where(df['Category'] == 'spam', -1, 1)
df.head(2)

,Category,Message,word_count,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,...,word_freq_conference,char_freq_;,char_freq_(,char_freq_[,char_freq_!,char_freq_$,char_freq_#,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,1,"Go until jurong point, crazy.. Available only ...",20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,3
1,1,Ok lar... Joking wif u oni...,6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1,2


In [4]:
experiment_name = "Default"
"""
metrics

cv_best_spam_f1
cv_mean_spam_f1
cv_std_spam_f1
test_roc_auc
test_accuracy
precision
recall
f1 score
weighted f1 score
"""
metric_to_sort_by = "cv_best_spam_f1"
ascending = False

experiment = mlflow.get_experiment_by_name(experiment_name)
experiment_id = experiment.experiment_id

runs_df = mlflow.search_runs(experiment_ids=[experiment_id])
runs_df = runs_df[runs_df["status"] == "FINISHED"]

best_run = (
    runs_df
    .sort_values(by=f"metrics.{metric_to_sort_by}", ascending=ascending)
    .iloc[0]
)

best_run_id = best_run["run_id"]
print("Best run ID:", best_run_id)
print("Run name:", best_run["tags.mlflow.runName"])
print(f"Best {metric_to_sort_by}:", best_run[f"metrics.{metric_to_sort_by}"])

Best run ID: b7a3b0360aac4a4f83eb7debc983a0a3
Run name: IsolationForest_BayesianSearch_AllConfigs
Best cv_best_spam_f1: 0.46286180101832


In [5]:
model_uri = f"runs:/{best_run_id}/best_model"
y = df["Category"]
X_new = df.drop(["Category", "Message", "word_count"], axis=1)
real_messages = df["Category"] == 1
spam_messages = df["Category"] == -1
y_real = y[real_messages]
y_spam = y[spam_messages]
X_real = X_new[real_messages]
X_spam = X_new[spam_messages]

loaded_model = mlflow.sklearn.load_model(model_uri)

predictions_all = loaded_model.predict(X_new)
predictions_real = loaded_model.predict(X_real)
predictions_spam = loaded_model.predict(X_spam)

In [6]:
from sklearn.metrics import accuracy_score

accuracy_all = accuracy_score(y, predictions_all)
accuracy_real = accuracy_score(y_real, predictions_real)
accuracy_spam = accuracy_score(y_spam, predictions_spam)
print(f"Accuracy metrics on {datafile}")
print(f"Accuracy on all messages: {accuracy_all:.2%}")
print(f"Accuracy on real messages: {accuracy_real:.2%}")
print(f"Accuracy on spam messages: {accuracy_spam:.2%}")


Accuracy metrics on SMSSpamCollection.csv
Accuracy on all messages: 82.02%
Accuracy on real messages: 86.03%
Accuracy on spam messages: 56.09%


In [7]:
datafile = "spambase.csv"

df = pd.read_csv(f"data/{datafile}")
# newdata = dc.Data_class(data = df, )
# df = newdata.data
df["Category"] = np.where(df['Category'] == 1, -1, 1)


In [8]:
y = df["Category"]
X_new = df.drop(["Category",], axis=1)
real_messages = df["Category"] == 1
spam_messages = df["Category"] == -1
y_real = y[real_messages]
y_spam = y[spam_messages]
X_real = X_new[real_messages]
X_spam = X_new[spam_messages]

predictions_all = loaded_model.predict(X_new)
predictions_real = loaded_model.predict(X_real)
predictions_spam = loaded_model.predict(X_spam)

In [9]:
accuracy_all = accuracy_score(y, predictions_all)
accuracy_real = accuracy_score(y_real, predictions_real)
accuracy_spam = accuracy_score(y_spam, predictions_spam)
print(f"Accuracy metrics on {datafile}")
print(f"Accuracy on all messages: {accuracy_all:.2%}")
print(f"Accuracy on real messages: {accuracy_real:.2%}")
print(f"Accuracy on spam messages: {accuracy_spam:.2%}")


Accuracy metrics on spambase.csv
Accuracy on all messages: 69.25%
Accuracy on real messages: 61.66%
Accuracy on spam messages: 80.92%


In [10]:
datafile = "hf-Marketing.csv"

df = pd.read_csv(f"data/{datafile}", header= None)
df.columns = ["Message"]

newdata = dc.Data_class(data = df, )
df = newdata.data
df["Category"] = -1

In [11]:
y = df["Category"]
X_new = df.drop(["Category", "Message", "word_count"], axis=1)

predictions_all = loaded_model.predict(X_new)


In [12]:

accuracy_spam = accuracy_score(y, predictions_all)
print(f"Accuracy metrics on {datafile}")

print(f"Accuracy on spam messages: {accuracy_spam:.2%}")

Accuracy metrics on hf-Marketing.csv
Accuracy on spam messages: 88.45%
